# Results for model: mistralai/mistral-nemotron

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np

# Load data
df = pl.scan_parquet('./jane_street_data/train.parquet')

# Calculate global quantile
quantile_threshold = df.select([
    pl.col('feature_00'),
    pl.col('responder_6')]
).group_by('responder_6').agg([
    pl.col('feature_00').quantile(0.85, None).suffix('_q85')
]).collect()

# Feature engineering
processed_df = df.join(
    quantile_threshold,
    on='responder_6',
    how='left'
).with_columns([
    (pl.col('feature_00_q85') / pl.col('feature_00_q85').max()).alias('top_quantile_feature'),
    (pl.col('responder_6') != pl.col('responder_6').max()).alias('is_respondent_active')
])

# Prepare training data
train_X = processed_df.filter(pl.col('date_id') <= 0).select([
    pl.col('top_quantile_feature'),
    pl.col('is_respondent_active'),
    pl.col('feature_00').alias('X0'),
    pl.col('feature_01').alias('X1'),
    pl.col('feature_02').alias('X2'),
    pl.col('feature_03').alias('X3'),
    pl.col('feature_04').alias('X4'),
    pl.col('feature_05').alias('X5'),
    pl.col('feature_06').alias('X6'),
    pl.col('feature_07').alias('X7'),
    pl.col('feature_08').alias('X8'),
    pl.col('feature_09').alias('X9'),
    pl.col('feature_10').alias('X10'),
    pl.col('feature_11').alias('X11'),
    pl.col('feature_12').alias('X12'),
    pl.col('feature_13').alias('X13'),
    pl.col('feature_14').alias('X14'),
    pl.col('feature_15').alias('X15'),
    pl.col('feature_16').alias('X16'),
    pl.col('feature_17').alias('X17'),
    pl.col('feature_18').alias('X18'),
    pl.col('feature_19').alias('X19'),
    pl.col('feature_20').alias('X20'),
    pl.col('feature_21').alias('X21'),
    pl.col('feature_22').alias('X22'),
    pl.col('feature_23').alias('X23'),
    pl.col('feature_24').alias('X24'),
    pl.col('feature_25').alias('X25'),
    pl.col('feature_26').alias('X26'),
    pl.col('feature_27').alias('X27'),
    pl.col('feature_28').alias('X28'),
    pl.col('feature_29').alias('X29'),
    pl.col('feature_30').alias('X30'),
    pl.col('feature_31').alias('X31'),
    pl.col('feature_32').alias('X32'),
    pl.col('feature_33').alias('X33'),
    pl.col('feature_34').alias('X34'),
    pl.col('feature_35').alias('X35'),
    pl.col('feature_36').alias('X36'),
    pl.col('feature_37').alias('X37'),
    pl.col('feature_38').alias('X38'),
    pl.col('feature_39').alias('X39'),
    pl.col('feature_40').alias('X40'),
    pl.col('feature_41').alias('X41')
]).collect().to_pandas()

train_y = processed_df.filter(pl.col('date_id') <= 0).select(pl.col('responder_6')).collect().to_pandas()

# Split data for validation
split_idx = int(0.8 * len(train_X))
X_train, X_val = train_X.iloc[:split_idx], train_X.iloc[split_idx:]
y_train, y_val = train_y.iloc[:split_idx], train_y.iloc[split_idx:]

# Train model
model = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.